# Traffic Dataset Cleaning + MySQL Database



In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("traffic.csv")

print("Shape:", df.shape)
display(df.head())


Shape: (5000, 15)


,Timestamp,Latitude,Longitude,Vehicle_Count,Traffic_Speed_kmh,Road_Occupancy_%,Traffic_Light_State,Weather_Condition,Accident_Report,Sentiment_Score,Ride_Sharing_Demand,Parking_Availability,Emission_Levels_g_km,Energy_Consumption_L_h,Traffic_Condition
0,01-03-2024 00:00,40.842275,-73.703149,205,49.893435,82.652780,Yellow,Clear,0,-0.609199,2,45,450.760055,19.574337,High
1,01-03-2024 00:05,40.831119,-73.987354,202,22.383965,45.829298,Green,Clear,0,0.965442,16,1,321.800341,5.385554,High
2,01-03-2024 00:10,40.819549,-73.732462,252,46.889699,82.772465,Green,Rain,0,0.289660,16,49,231.152655,10.277477,High
3,01-03-2024 00:15,40.725849,-73.980134,37,5.730536,37.695567,Red,Fog,0,-0.271965,66,10,410.384292,29.243279,High
4,01-03-2024 00:20,40.813265,-73.961631,64,61.348034,22.313358,Red,Snow,0,-0.797606,3,5,364.466342,16.801459,Low


In [2]:
# Inspect the dataset
print(df.info())
print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique categorical values:")
for col in df.select_dtypes(include="object").columns:
    print(col, ":", df[col].unique())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Timestamp               5000 non-null   object 
 1   Latitude                5000 non-null   float64
 2   Longitude               5000 non-null   float64
 3   Vehicle_Count           5000 non-null   int64  
 4   Traffic_Speed_kmh       5000 non-null   float64
 5   Road_Occupancy_%        5000 non-null   float64
 6   Traffic_Light_State     5000 non-null   object 
 7   Weather_Condition       5000 non-null   object 
 8   Accident_Report         5000 non-null   int64  
 9   Sentiment_Score         5000 non-null   float64
 10  Ride_Sharing_Demand     5000 non-null   int64  
 11  Parking_Availability    5000 non-null   int64  
 12  Emission_Levels_g_km    5000 non-null   float64
 13  Energy_Consumption_L_h  5000 non-null   float64
 14  Traffic_Condition       5000 non-null   

Timestamp                 0
Latitude                  0
Longitude                 0
Vehicle_Count             0
Traffic_Speed_kmh         0
Road_Occupancy_%          0
Traffic_Light_State       0
Weather_Condition         0
Accident_Report           0
Sentiment_Score           0
Ride_Sharing_Demand       0
Parking_Availability      0
Emission_Levels_g_km      0
Energy_Consumption_L_h    0
Traffic_Condition         0
dtype: int64


Duplicate rows: 0

Unique categorical values:
Timestamp : ['01-03-2024 00:00' '01-03-2024 00:05' '01-03-2024 00:10' ...
 '18-03-2024 08:25' '18-03-2024 08:30' '18-03-2024 08:35']
Traffic_Light_State : ['Yellow' 'Green' 'Red']
Weather_Condition : ['Clear' 'Rain' 'Fog' 'Snow']
Traffic_Condition : ['High' 'Low' 'Medium']


In [3]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("%", "pct", regex=False)
    .str.replace("/", "_per_", regex=False)
    .str.replace(r"[^a-z0-9_]+", "_", regex=True)
    .str.strip("_")
)

print(df.columns.tolist())


['timestamp', 'latitude', 'longitude', 'vehicle_count', 'traffic_speed_kmh', 'road_occupancy_pct', 'traffic_light_state', 'weather_condition', 'accident_report', 'sentiment_score', 'ride_sharing_demand', 'parking_availability', 'emission_levels_g_km', 'energy_consumption_l_h', 'traffic_condition']


In [4]:
# Convert timestamp
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

# Clean text columns
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Remove invalid timestamps
df = df.dropna(subset=["timestamp"]).reset_index(drop=True)

print("Rows after basic cleaning:", len(df))


Rows after basic cleaning: 5000


In [5]:
# Validate numeric ranges
valid = (
    df["vehicle_count"].between(0, 100000)
    & df["traffic_speed_kmh"].between(0, 300)
    & df["road_occupancy_pct"].between(0, 100)
    & df["accident_report"].isin([0, 1])
    & df["sentiment_score"].between(-1, 1)
    & df["ride_sharing_demand"].between(0, 100000)
    & df["parking_availability"].between(0, 100000)
    & (df["emission_levels_g_km"] >= 0)
    & (df["energy_consumption_l_h"] >= 0)
)

print("Invalid rows:", (~valid).sum())

df = df.loc[valid].reset_index(drop=True)

for col in ["traffic_light_state", "weather_condition", "traffic_condition"]:
    df[col] = df[col].str.title()

print("Final shape:", df.shape)
display(df.head())


Invalid rows: 0
Final shape: (5000, 15)


,timestamp,latitude,longitude,vehicle_count,traffic_speed_kmh,road_occupancy_pct,traffic_light_state,weather_condition,accident_report,sentiment_score,ride_sharing_demand,parking_availability,emission_levels_g_km,energy_consumption_l_h,traffic_condition
0,2024-03-01 00:00:00,40.842275,-73.703149,205,49.893435,82.652780,Yellow,Clear,0,-0.609199,2,45,450.760055,19.574337,High
1,2024-03-01 00:05:00,40.831119,-73.987354,202,22.383965,45.829298,Green,Clear,0,0.965442,16,1,321.800341,5.385554,High
2,2024-03-01 00:10:00,40.819549,-73.732462,252,46.889699,82.772465,Green,Rain,0,0.289660,16,49,231.152655,10.277477,High
3,2024-03-01 00:15:00,40.725849,-73.980134,37,5.730536,37.695567,Red,Fog,0,-0.271965,66,10,410.384292,29.243279,High
4,2024-03-01 00:20:00,40.813265,-73.961631,64,61.348034,22.313358,Red,Snow,0,-0.797606,3,5,364.466342,16.801459,Low


In [6]:
# Final quality check
print("Missing values:")
display(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())

display(df.describe())


Missing values:


timestamp                 0
latitude                  0
longitude                 0
vehicle_count             0
traffic_speed_kmh         0
road_occupancy_pct        0
traffic_light_state       0
weather_condition         0
accident_report           0
sentiment_score           0
ride_sharing_demand       0
parking_availability      0
emission_levels_g_km      0
energy_consumption_l_h    0
traffic_condition         0
dtype: int64

Duplicate rows: 0


,timestamp,latitude,longitude,vehicle_count,traffic_speed_kmh,road_occupancy_pct,accident_report,sentiment_score,ride_sharing_demand,parking_availability,emission_levels_g_km,energy_consumption_l_h
count,5000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,2024-03-09 16:17:30,40.749645,-73.847433,153.921200,42.111096,54.748397,0.096000,-0.005652,50.487800,24.613000,272.174927,17.343243
min,2024-03-01 00:00:00,40.600016,-73.999987,10.000000,5.002789,10.005031,0.000000,-0.999819,1.000000,0.000000,50.136855,5.003787
25%,2024-03-05 08:08:45,40.675403,-73.920281,80.000000,23.195752,32.089653,0.000000,-0.500154,26.000000,12.000000,160.564433,11.098761
50%,2024-03-09 16:17:30,40.748875,-73.846058,156.000000,42.191599,54.657297,0.000000,-0.010564,50.000000,24.000000,272.045513,17.153791
75%,2024-03-14 00:26:15,40.824735,-73.771685,226.000000,60.751760,77.581720,0.000000,0.500518,75.000000,37.000000,382.242055,23.516595
max,2024-03-18 08:35:00,40.899972,-73.700159,299.000000,79.997556,99.999729,1.000000,0.999354,99.000000,49.000000,499.922663,29.995416
std,NaN,0.086123,0.086204,83.523342,21.707720,26.145238,0.294621,0.583790,28.484426,14.532511,130.086372,7.208277


In [20]:
# Save cleaned CSV
df.to_csv("cleaned_data.csv", index=False)
print("Cleaned dataset saved.")

Cleaned dataset saved.


In [8]:
# Install MySQL connector if needed
# Run this once:
# !pip install mysql-connector-python


In [9]:
import mysql.connector

# Change these values according to your MySQL installation
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Saxena@123"   
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS traffic_db")
cursor.execute("USE traffic_db")

print("Database traffic_db is ready.")


Database traffic_db is ready.


In [13]:
# Check the current timestamp type
print(df["timestamp"].dtype)
print(type(df["timestamp"].iloc[0]))
print(df["timestamp"].iloc[0])

datetime64[ns]
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
2024-03-01 00:00:00


In [14]:
# Convert timestamp to MySQL-compatible string
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["timestamp"] = df["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")

print(type(df["timestamp"].iloc[0]))
print(df["timestamp"].iloc[0])

<class 'str'>
2024-03-01 00:00:00


In [15]:
# ==========================================
# INSERT CLEANED DATA INTO MYSQL
# ==========================================

insert_query = '''
INSERT INTO traffic_data (
    timestamp,
    latitude,
    longitude,
    vehicle_count,
    traffic_speed_kmh,
    road_occupancy_pct,
    traffic_light_state,
    weather_condition,
    accident_report,
    sentiment_score,
    ride_sharing_demand,
    parking_availability,
    emission_levels_g_km,
    energy_consumption_l_h,
    traffic_condition
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
'''

data = []

for _, row in df.iterrows():

    values = (
        str(row["timestamp"]),
        float(row["latitude"]),
        float(row["longitude"]),
        int(row["vehicle_count"]),
        float(row["traffic_speed_kmh"]),
        float(row["road_occupancy_pct"]),
        str(row["traffic_light_state"]),
        str(row["weather_condition"]),
        int(row["accident_report"]),
        float(row["sentiment_score"]),
        int(row["ride_sharing_demand"]),
        int(row["parking_availability"]),
        float(row["emission_levels_g_km"]),
        float(row["energy_consumption_l_h"]),
        str(row["traffic_condition"])
    )

    data.append(values)


# Insert all records
cursor.executemany(insert_query, data)

# Save changes
conn.commit()

print(cursor.rowcount, "rows inserted successfully.")

5000 rows inserted successfully.


In [16]:
# Insert cleaned data into MySQL
insert_query = '''
INSERT INTO traffic_data (
    timestamp,
    latitude,
    longitude,
    vehicle_count,
    traffic_speed_kmh,
    road_occupancy_pct,
    traffic_light_state,
    weather_condition,
    accident_report,
    sentiment_score,
    ride_sharing_demand,
    parking_availability,
    emission_levels_g_km,
    energy_consumption_l_h,
    traffic_condition
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
'''

data = [
    tuple(row)
    for row in df[
        [
            "timestamp", "latitude", "longitude", "vehicle_count",
            "traffic_speed_kmh", "road_occupancy_pct",
            "traffic_light_state", "weather_condition", "accident_report",
            "sentiment_score", "ride_sharing_demand", "parking_availability",
            "emission_levels_g_km", "energy_consumption_l_h",
            "traffic_condition"
        ]
    ].itertuples(index=False, name=None)
]

cursor.executemany(insert_query, data)
conn.commit()

print(cursor.rowcount, "rows inserted.")


5000 rows inserted.


In [17]:
# Verify data in MySQL
cursor.execute("SELECT COUNT(*) FROM traffic_data")
print("Total rows:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM traffic_data LIMIT 10")
rows = cursor.fetchall()

for row in rows:
    print(row)


Total rows: 10000
(1, datetime.datetime(2024, 3, 1, 0, 0), Decimal('40.8422753'), Decimal('-73.7031487'), 205, Decimal('49.893'), Decimal('82.653'), 'Yellow', 'Clear', 0, Decimal('-0.60920'), 2, 45, Decimal('450.760'), Decimal('19.574'), 'High')
(2, datetime.datetime(2024, 3, 1, 0, 5), Decimal('40.8311194'), Decimal('-73.9873541'), 202, Decimal('22.384'), Decimal('45.829'), 'Green', 'Clear', 0, Decimal('0.96544'), 16, 1, Decimal('321.800'), Decimal('5.386'), 'High')
(3, datetime.datetime(2024, 3, 1, 0, 10), Decimal('40.8195488'), Decimal('-73.7324619'), 252, Decimal('46.890'), Decimal('82.772'), 'Green', 'Rain', 0, Decimal('0.28966'), 16, 49, Decimal('231.153'), Decimal('10.277'), 'High')
(4, datetime.datetime(2024, 3, 1, 0, 15), Decimal('40.7258489'), Decimal('-73.9801341'), 37, Decimal('5.731'), Decimal('37.696'), 'Red', 'Fog', 0, Decimal('-0.27196'), 66, 10, Decimal('410.384'), Decimal('29.243'), 'High')
(5, datetime.datetime(2024, 3, 1, 0, 20), Decimal('40.8132647'), Decimal('-73.9

In [18]:
# Close connection
cursor.close()
conn.close()

print("MySQL connection closed.")


MySQL connection closed.
